# 2.2. Evaluate image quality assessment metrics on ablated images generated from 2.1 against their raw reference

Metrics included in this notebook
- MAE
- SSIM
- PSNR
- LPIPS
- DISTS
- foreground aware SSIM and PSNR (see `metrics.py` implementation)



In [1]:
import pathlib
import yaml
from typing import Dict

import torch
# from torch.utils.data import DataLoader
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

from image_ablation_analysis.indexing import ParquetIndex
from image_ablation_analysis.hooks.normalization import BitDepthNormalizer
from image_ablation_analysis.eval.metrics import (
    MetricSpec,
    to_rgb_space,
    identity,
    functional_dists,
    functional_mae,
    functional_psnr
)
from image_ablation_analysis.eval.masked_metrics import (
    ForegroundPSNR,
    ForegroundSSIM,
)
from image_ablation_analysis.eval.eval_utils import ImagePairDataset
from image_ablation_analysis.eval.eval_runner import EvalRunner

## Pathing

In [2]:
module_config_path = pathlib.Path("..") / '2.metrics_ablation_analysis' / 'config.yml'
if not module_config_path.exists():
    raise FileNotFoundError(f"Module config file not found: {module_config_path}")
config = yaml.safe_load(module_config_path.read_text())

abl_root = pathlib.Path(config['ablation_output_path']).resolve(strict=True)

out_dir = abl_root / "results" / "metrics"
out_dir.mkdir(parents=True, exist_ok=True)

abl_images = list(abl_root.rglob("*.tiff"))
if not abl_images:
    raise FileNotFoundError(f"No ablated images found in {abl_root}.")
else:
    print(f"Found {len(abl_images)} ablated images in {abl_root}.")

Found 2268612 ablated images in /mnt/hdd20tb/alsf_ablation.


## Set cuda device used for accelarating metrics computation

In [3]:
# Prefer the second GPU for this evaluation analysis by PCIE order
# I had my secondary GPU installed on the second slot, adjust as needed per your setup

if torch.cuda.is_available():
    n_devices = torch.cuda.device_count()
    print(f"Number of available CUDA devices: {n_devices}")
    for i in range(n_devices):
        device_name = torch.cuda.get_device_name(i)
        print(f"CUDA Device {i}: {device_name}")    
    
    if n_devices >= 2:
        # Attempt using the second GPU (index 1) in PCIe order
        device = torch.device('cuda:1')
        print(f"Selected device: {torch.cuda.get_device_name(device.index)}")
    else:
        # fallback to the first GPU if only one is available
        device = torch.device('cuda:0') 
        print(f"Only one CUDA device available. Using device: {torch.cuda.get_device_name(device.index)}")
else:
    raise RuntimeError("No CUDA devices available.")

print(f"Using device: {device}")

Number of available CUDA devices: 2
CUDA Device 0: NVIDIA RTX A6000
CUDA Device 1: NVIDIA GeForce RTX 3090
Selected device: NVIDIA GeForce RTX 3090
Using device: cuda:1


## Define metric functions

In [4]:
# Due to the inconsistent way certain metrics support/handle aggregation
# some metrics need special wrappers that ensures metrics are computed per
# image (sample) within the same batch. These wrappers are imported as
# functional_{metric_name} here and used. 
metric_fns = {
    "mae": functional_mae,
    "ssim": StructuralSimilarityIndexMeasure(data_range=1.0, reduction="none").to(device),
    "psnr": functional_psnr,
    "lpips": LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True, reduction="none").to(device),
    "dists": functional_dists,
    "foreground_ssim": ForegroundSSIM(
        kernel_size=11,
        data_range=(0.0, 1.0),
        eps=1e-12,
        dtype=torch.float32
    ).to(device),
    "foreground_psnr": ForegroundPSNR(
        data_range=(0.0, 1.0),
        eps=1e-12,
        dtype=torch.float32
    ).to(device)
}

# Deep learning metrics DISTS and LPIPS are trained on RGB images and thus 
# needs a special preprocessing step to duplicate the grayscale channels
# to RGB channels. Other metrics can use the identity function.
METRICS: Dict[str, MetricSpec] = {
    "mae": MetricSpec(
        fn=metric_fns["mae"],
        preprocess=identity,
    ),
    "ssim": MetricSpec(
        fn=metric_fns["ssim"],
        preprocess=identity,
    ),
    "psnr": MetricSpec(
        fn=metric_fns["psnr"],
        preprocess=identity,
    ),
    "lpips": MetricSpec(
        fn=metric_fns["lpips"],
        preprocess=to_rgb_space,
    ),
    "dists": MetricSpec(
        fn=metric_fns["dists"],
        preprocess=to_rgb_space,
    ),
    "foreground_ssim": MetricSpec(
        fn=metric_fns["foreground_ssim"],
        preprocess=identity,
    ),
    "foreground_psnr": MetricSpec(
        fn=metric_fns["foreground_psnr"],
        preprocess=identity,
    ),
}

## Checking the generated ablated images

In [5]:
index = ParquetIndex(index_dir=abl_root / "ablated_index")
index_df = index.read()
index_df.head()

,created_at,run_id,original_abs_path,original_rel_path,aug_abs_path,aug_rel_path,variant,config_id,params_json,param_fixed,...,Metadata_AbsPositionZ,Metadata_ChannelID,Metadata_Col,Metadata_FieldID,Metadata_PlaneID,Metadata_PositionX,Metadata_PositionY,Metadata_PositionZ,Metadata_Row,Metadata_Reimaged
0,20260218T201130562139Z,158233ca-0f56-414c-9c0c-1039f01abf46,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,"xform_abl_distort=(0.1,5)_8b45fe533dc58654",albumentations:GridDistortion:8b45fe533dc58654,"{""backend"":""albumentations"",""transform_name"":""...","[""num_steps""]",...,0.134342,6.0,13.0,1.0,1.0,0.0,0.0,-0.000006,13.0,False
1,20260218T201130562139Z,158233ca-0f56-414c-9c0c-1039f01abf46,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,"xform_abl_distort=(0.2,5)_eabce0e6ac52da0e",albumentations:GridDistortion:eabce0e6ac52da0e,"{""backend"":""albumentations"",""transform_name"":""...","[""num_steps""]",...,0.134342,6.0,13.0,1.0,1.0,0.0,0.0,-0.000006,13.0,False
2,20260218T201130562139Z,158233ca-0f56-414c-9c0c-1039f01abf46,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,"xform_abl_distort=(0.4,5)_cc7c9f196649cf8a",albumentations:GridDistortion:cc7c9f196649cf8a,"{""backend"":""albumentations"",""transform_name"":""...","[""num_steps""]",...,0.134342,6.0,13.0,1.0,1.0,0.0,0.0,-0.000006,13.0,False
3,20260218T201130562139Z,158233ca-0f56-414c-9c0c-1039f01abf46,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,"xform_abl_distort=(0.6,5)_82d5d02df8b0a3ac",albumentations:GridDistortion:82d5d02df8b0a3ac,"{""backend"":""albumentations"",""transform_name"":""...","[""num_steps""]",...,0.134342,6.0,13.0,1.0,1.0,0.0,0.0,-0.000006,13.0,False
4,20260218T201130562139Z,158233ca-0f56-414c-9c0c-1039f01abf46,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,"xform_abl_distort=(0.8,5)_0757b3cfde079b6b",albumentations:GridDistortion:0757b3cfde079b6b,"{""backend"":""albumentations"",""transform_name"":""...","[""num_steps""]",...,0.134342,6.0,13.0,1.0,1.0,0.0,0.0,-0.000006,13.0,False


## Setting up the evaluation dataset from the ablated image index
The dataset would return pairs of ablatied image and the non-ablated raw version for convenient metric computation

In [6]:
# normalize 16-bit ablated images to [0, 1] float32
# this helps the evaluation with metrics that assume inputs are in [0, 1]
# like PSNR that is configured with data_range=1.0
normalizer = BitDepthNormalizer(bit_depth=16)

dataset = ImagePairDataset(
    # This image pair dataset takes the index dataframe from 2.1.generate_ablation
    # that tracks every single saved ablated image and its corresponding original image.
    # and returns pairs of (original, ablated) images for convenient evaluation
    index_df, 
    # this makes the dataset class call the normalizer 
    # before finally yielding the image pair
    # please see the ImagePairDataset implementation for details
    normalizer,  
    metadata_cols=[
        # append these metadata to output metrics eval for more 
        # convenient analysis downstream
        'variant', 'run_id', 'config_id', 
        'Metadata_Plate', 'cell_line', 'seeding_density',
        'original_abs_path', 'aug_abs_path',
        'param_fixed', 'param_swept', 'param_values'
    ]
)

## Invoke metrics evaluation runner

In [7]:
runner = EvalRunner(
    index_df=index_df,
    normalizer=normalizer,
)

runner.run(
    out_dir=out_dir,
    metrics=METRICS,
    device=device,
    batch_size=8,
    num_workers=4,
    force_overwrite=True,
)

Evaluating metrics:   0%|          | 0/290493 [00:00<?, ?it/s]